Proses Data labelling otomatis dengan indoBERT

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

BASE_DIR = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method'

# --- 1. LOAD DATA BERLABEL LAMA ---
df = pd.read_csv('03_data_labelling/data_labeling.csv')
print(f"Total data berlabel: {len(df)}")
print(f"\nDistribusi label:")
print(df['label_pks'].value_counts())

# --- 2. MAPPING LABEL KE INTEGER ---
label2id = {'keluhan': 0, 'saran': 1, 'pujian': 2}
id2label = {0: 'keluhan', 1: 'saran', 2: 'pujian'}

df['label_id'] = df['label_pks'].map(label2id)

# --- 3. SPLIT TRAIN/VAL 80/20 ---
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label_id']  # Pastikan proporsi kelas seimbang
)

print(f"\nData training   : {len(df_train)}")
print(f"Data validasi   : {len(df_val)}")
print(f"\nDistribusi training:")
print(df_train['label_pks'].value_counts())
print(f"\nDistribusi validasi:")
print(df_val['label_pks'].value_counts())

Total data berlabel: 18534

Distribusi label:
label_pks
keluhan    10604
pujian      5116
saran       2814
Name: count, dtype: int64

Data training   : 14827
Data validasi   : 3707

Distribusi training:
label_pks
keluhan    8483
pujian     4093
saran      2251
Name: count, dtype: int64

Distribusi validasi:
label_pks
keluhan    2121
pujian     1023
saran       563
Name: count, dtype: int64


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

MODEL_NAME = "indobenchmark/indobert-base-p1"

print(f"Loading tokenizer & model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- FIX: Override config terlebih dahulu ---
config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = 3
config.id2label   = {0: 'keluhan', 1: 'saran', 2: 'pujian'}
config.label2id   = {'keluhan': 0, 'saran': 1, 'pujian': 2}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True  # Abaikan mismatch ukuran classifier head
)

# Pindahkan model ke GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

print(f"\n✅ Model berhasil dimuat!")
print(f"   Jumlah label : {model.config.num_labels}")
print(f"   Label        : {model.config.id2label}")
print(f"   Device       : {device}")
print(f"   GPU          : {torch.cuda.get_device_name(0)}")

Loading tokenizer & model: indobenchmark/indobert-base-p1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Model berhasil dimuat!
   Jumlah label : 3
   Label        : {0: 'keluhan', 1: 'saran', 2: 'pujian'}
   Device       : cuda
   GPU          : NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# Cek kolom yang tersedia di df_train
print("Kolom tersedia:", df_train.columns.tolist())
print("\nSampel data:")
print(df_train.head(2))

Kolom tersedia: ['id', 'timestamp', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 'ownerUsername', 'text', 'label_pks', 'label_id']

Sampel data:
                 id                  timestamp  likesCount  \
846    1.810000e+16  2025-10-14 10:18:34+00:00           0   
10662  1.810000e+16  2025-03-04 09:28:00+00:00           0   

                                                 postUrl  \
846    https://www.instagram.com/ditjenpajakri/p/DPQj...   
10662  https://www.instagram.com/ditjenpajakri/p/DGw2...   

                                              commentUrl  \
846    https://www.instagram.com/ditjenpajakri/p/DPQj...   
10662  https://www.instagram.com/ditjenpajakri/p/DGw2...   

                                   source_file ownerUsername  \
846    instagram_comments_10.22.2025_18.55.csv   fajarnur212   
10662  instagram_comments_10.22.2025_20.32.csv    irmaaa_una   

                                                    text label_pks  label_id  
846                      

In [ ]:
from torch.utils.data import Dataset

MAX_LENGTH = 128  # Panjang maksimum token

class KomentarDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Buat dataset
train_dataset = KomentarDataset(
    texts=df_train['text'].tolist(),
    labels=df_train['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = KomentarDataset(
    texts=df_val['text'].tolist(),
    labels=df_val['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print(f"Train dataset : {len(train_dataset)} sampel")
print(f"Val dataset   : {len(val_dataset)} sampel")

Train dataset : 14827 sampel
Val dataset   : 3707 sampel


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score
import numpy as np

# --- KONFIGURASI ---
EPOCHS      = 5
BATCH_SIZE  = 32   # Sesuai VRAM RTX 4060 8GB
LEARNING_RATE = 2e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# --- TRAINING LOOP ---
print("="*60)
print("MULAI FINE-TUNING INDOBERT")
print("="*60)
print(f"Epochs      : {EPOCHS}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Total steps : {total_steps}")
print("-"*60)

best_f1    = 0
best_epoch = 0

for epoch in range(EPOCHS):
    # === TRAINING ===
    model.train()
    total_loss   = 0
    train_preds  = []
    train_labels = []

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(labels.cpu().numpy())

    avg_train_loss = total_loss / len(train_loader)
    train_f1       = f1_score(train_labels, train_preds, average='macro')

    # === VALIDASI ===
    model.eval()
    val_preds  = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(labels.cpu().numpy())

    val_f1 = f1_score(val_labels, val_preds, average='macro')

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {avg_train_loss:.4f} | Train F1 : {train_f1:.4f}")
    print(f"  Val F1     : {val_f1:.4f}")
    print(f"  Classification Report (Validasi):")
    print(classification_report(
        val_labels, val_preds,
        target_names=['keluhan', 'saran', 'pujian']
    ))

    # Simpan model terbaik
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch + 1
        model.save_pretrained(os.path.join(BASE_DIR, 'indobert_finetuned'))
        tokenizer.save_pretrained(os.path.join(BASE_DIR, 'indobert_finetuned'))
        print(f"  💾 Model terbaik disimpan! (F1: {best_f1:.4f})")

print("\n" + "="*60)
print(f"FINE-TUNING SELESAI!")
print(f"Best Val F1 : {best_f1:.4f} (Epoch {best_epoch})")
print("="*60)

MULAI FINE-TUNING INDOBERT
Epochs      : 5
Batch size  : 32
Total steps : 2320
------------------------------------------------------------

Epoch 1/5
  Train Loss : 0.4281 | Train F1 : 0.7668
  Val F1     : 0.8561
  Classification Report (Validasi):
              precision    recall  f1-score   support

     keluhan       0.90      0.94      0.92      2121
       saran       0.79      0.65      0.71       563
      pujian       0.94      0.93      0.94      1023

    accuracy                           0.89      3707
   macro avg       0.87      0.84      0.86      3707
weighted avg       0.89      0.89      0.89      3707



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  💾 Model terbaik disimpan! (F1: 0.8561)

Epoch 2/5
  Train Loss : 0.2387 | Train F1 : 0.8875
  Val F1     : 0.8593
  Classification Report (Validasi):
              precision    recall  f1-score   support

     keluhan       0.92      0.91      0.91      2121
       saran       0.73      0.72      0.72       563
      pujian       0.93      0.95      0.94      1023

    accuracy                           0.89      3707
   macro avg       0.86      0.86      0.86      3707
weighted avg       0.89      0.89      0.89      3707



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  💾 Model terbaik disimpan! (F1: 0.8593)

Epoch 3/5
  Train Loss : 0.1434 | Train F1 : 0.9416
  Val F1     : 0.8522
  Classification Report (Validasi):
              precision    recall  f1-score   support

     keluhan       0.91      0.92      0.91      2121
       saran       0.72      0.69      0.71       563
      pujian       0.94      0.93      0.94      1023

    accuracy                           0.89      3707
   macro avg       0.86      0.85      0.85      3707
weighted avg       0.89      0.89      0.89      3707


Epoch 4/5
  Train Loss : 0.0841 | Train F1 : 0.9726
  Val F1     : 0.8566
  Classification Report (Validasi):
              precision    recall  f1-score   support

     keluhan       0.92      0.90      0.91      2121
       saran       0.71      0.73      0.72       563
      pujian       0.93      0.95      0.94      1023

    accuracy                           0.89      3707
   macro avg       0.85      0.86      0.86      3707
weighted avg       0.89      0

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

# --- LOAD MODEL TERBAIK ---
print("Loading model terbaik...")
model_path = os.path.join(BASE_DIR, 'indobert_finetuned')
tokenizer  = AutoTokenizer.from_pretrained(model_path)
model      = AutoModelForSequenceClassification.from_pretrained(model_path)
model      = model.to(device)
model.eval()
print("✅ Model siap digunakan!")

# --- LOAD DATA BARU ---
df_baru = pd.read_csv(os.path.join(BASE_DIR, 'cleaned_data_baru.csv'))
print(f"Total data baru yang akan diprediksi: {len(df_baru)}")

# --- FUNGSI PREDIKSI BATCH ---
def predict_batch(texts, batch_size=64):
    all_preds  = []
    all_scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch = [str(t)[:512] for t in batch]

        encoding = tokenizer(
            batch,
            max_length=128,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )

        input_ids      = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=1)
            preds   = torch.argmax(probs, dim=1).cpu().numpy()
            scores  = probs.max(dim=1).values.cpu().numpy()

        all_preds.extend(preds)
        all_scores.extend(scores)

        # Checkpoint setiap 10000 data
        if (i + batch_size) % 10000 == 0 and i > 0:
            print(f"\n💾 Progress: {len(all_preds)}/{len(texts)} data")

    return all_preds, all_scores

# --- JALANKAN PREDIKSI ---
print("\nMemulai prediksi data baru...")
preds, scores = predict_batch(df_baru['text'].tolist())

df_baru['label_pks']  = [id2label[p] for p in preds]
df_baru['confidence'] = [round(s, 4) for s in scores]

# --- DISTRIBUSI HASIL ---
print("\n=== DISTRIBUSI LABEL DATA BARU ===")
print(df_baru['label_pks'].value_counts())
print(f"\nRata-rata confidence: {df_baru['confidence'].mean():.4f}")

# --- SIMPAN ---
output_path = os.path.join(BASE_DIR, 'labelled_data_baru.csv')
df_baru.to_csv(output_path, index=False)
print(f"\n✅ Data baru berlabel tersimpan: {output_path}")

Loading model terbaik...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5338.36it/s]


✅ Model siap digunakan!
Total data baru yang akan diprediksi: 61629

Memulai prediksi data baru...


 65%|██████▌   | 626/963 [01:13<00:42,  7.94it/s]


💾 Progress: 40000/61629 data


100%|██████████| 963/963 [02:02<00:00,  7.83it/s]



=== DISTRIBUSI LABEL DATA BARU ===
label_pks
keluhan    38125
pujian     12025
saran      11479
Name: count, dtype: int64

Rata-rata confidence: 0.8890

✅ Data baru berlabel tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\labelled_data_baru.csv


Proses penggabungan data manual dan otomatis

In [ ]:
import pandas as pd
import os

BASE_DIR = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'

# --- LOAD DATA LAMA BERLABEL ---
df_lama = pd.read_csv(os.path.join(BASE_DIR, 'data_labeling.csv'))
df_lama['confidence'] = 1.0  # Label manual → confidence = 1.0

# --- LOAD DATA BARU BERLABEL ---
df_baru = pd.read_csv(os.path.join(BASE_DIR, 'labelled_data_baru.csv'))

print(f"Data lama berlabel : {len(df_lama)}")
print(f"Data baru berlabel : {len(df_baru)}")

# --- SAMAKAN KOLOM ---
KOLOM = ['id', 'timestamp', 'ownerUsername', 'text', 
         'likesCount', 'postUrl', 'commentUrl', 'source_file', 'label_pks']

# Cek kolom yang tersedia
print(f"\nKolom data lama : {df_lama.columns.tolist()}")
print(f"Kolom data baru : {df_baru.columns.tolist()}")

Data lama berlabel : 18534
Data baru berlabel : 61629

Kolom data lama : ['id', 'timestamp', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 'ownerUsername', 'text', 'label_pks', 'confidence']
Kolom data baru : ['id', 'timestamp', 'ownerUsername', 'text', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 'label_pks', 'confidence']


In [ ]:
# --- SAMAKAN URUTAN KOLOM ---
KOLOM = ['id', 'timestamp', 'ownerUsername', 'text', 'likesCount', 
         'postUrl', 'commentUrl', 'source_file', 'label_pks', 'confidence']

df_lama = df_lama[KOLOM]
df_baru = df_baru[KOLOM]

# --- GABUNGKAN ---
df_final = pd.concat([df_lama, df_baru], ignore_index=True)

print("=== HASIL PENGGABUNGAN ===")
print(f"Data lama berlabel : {len(df_lama)}")
print(f"Data baru berlabel : {len(df_baru)}")
print(f"Total gabungan     : {len(df_final)}")

print(f"\n=== DISTRIBUSI LABEL FINAL ===")
print(df_final['label_pks'].value_counts())
print(f"\nProporsi:")
print(df_final['label_pks'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

print(f"\n=== RATA-RATA CONFIDENCE ===")
print(df_final.groupby('label_pks')['confidence'].mean().round(4))

# --- SIMPAN ---
output_path = os.path.join(BASE_DIR, 'labelled_data_final.csv')
df_final.to_csv(output_path, index=False)
print(f"\n✅ Data final tersimpan: {output_path}")

=== HASIL PENGGABUNGAN ===
Data lama berlabel : 18534
Data baru berlabel : 61629
Total gabungan     : 80163

=== DISTRIBUSI LABEL FINAL ===
label_pks
keluhan    48729
pujian     17141
saran      14293
Name: count, dtype: int64

Proporsi:
label_pks
keluhan    60.79%
pujian     21.38%
saran      17.83%
Name: proportion, dtype: object

=== RATA-RATA CONFIDENCE ===
label_pks
keluhan    0.9306
pujian     0.9313
saran      0.8405
Name: confidence, dtype: float64

✅ Data final tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\labelled_data_final.csv
